# Dictionary Lookup vs Joins in PySpark

In [1]:
from datetime import date, timedelta

import pyspark.sql.functions as F
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import DateType, StructType, StructField, StringType, IntegerType

In [2]:
spark = SparkSession.builder.appName("DictVsJoins").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/20 23:39:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Create sample data

In [3]:
NUM_HOSPITALS = 1000
NUM_DAYS = 30
MEDICAL_DEPARTMENTS = [
    "emergency",
    "cardiology",
    "neurology",
    "surgery",
    "pediatrics",
    "orthopedics",
    "gynecology",
    "dermatology",
]

In [4]:
schema = StructType(
    [
        StructField("hospital_id", StringType(), nullable=True),
        StructField("service_date", DateType(), nullable=True),
        StructField("medical_department", StringType(), nullable=True),
        StructField("budget_allocated", IntegerType(), nullable=True),
    ]
)

hospitals = [(f"HOS_{i:04d}",) for i in range(1, NUM_HOSPITALS + 1)]
df_hospitals = spark.createDataFrame(hospitals, ["hospital_id"])

dates = [(date(2025, 1, 1) + timedelta(days=i),) for i in range(NUM_DAYS)]
df_dates = spark.createDataFrame(dates, ["service_date"])

departments = [(dept,) for dept in MEDICAL_DEPARTMENTS]
df_departments = spark.createDataFrame(departments, ["medical_department"])

df_healthcare = df_hospitals.crossJoin(df_dates).crossJoin(df_departments).withColumn(
    "budget_allocated",
    F.abs((F.hash(F.concat_ws("_", *df_hospitals.columns)) % 500000) + 50000)
).cache()

df_healthcare.show(5)

+-----------+------------+------------------+----------------+
|hospital_id|service_date|medical_department|budget_allocated|
+-----------+------------+------------------+----------------+
|   HOS_0001|  2025-01-01|         emergency|          287374|
|   HOS_0001|  2025-01-02|         emergency|          287374|
|   HOS_0001|  2025-01-03|         emergency|          287374|
|   HOS_0002|  2025-01-01|         emergency|          163067|
|   HOS_0002|  2025-01-02|         emergency|          163067|
+-----------+------------+------------------+----------------+
only showing top 5 rows



## Dictionary Mapping

In [5]:
dept_mapping = {
    "emergency": "critical_care",
    "Cardiology": "critical_care",
    "Surgery": "critical_care"
}
mapping_expr = F.create_map([F.lit(x) for x in sum(dept_mapping.items(), ())])
df_dict = df_healthcare.withColumn(
    "cost_center", F.coalesce(mapping_expr[F.col("medical_department")], F.lit("other"))
)

## Dataframe join

In [6]:
df_mapping = spark.createDataFrame(
    [["emergency", "critical_care"], ["cardiology", "critical_care"], ["surgery", "critical_care"]],
    ["medical_department", "cost_center"]
)
df_join = df_healthcare.join(
    df_mapping, on="medical_department", how="left"
)

## Performance comparison

In [7]:
%timeit -n 50 -r 3 df_dict.count()
%timeit -n 50 -r 3 df_join.count()

1.11 s ± 131 ms per loop (mean ± std. dev. of 3 runs, 50 loops each)


2.3 s ± 159 ms per loop (mean ± std. dev. of 3 runs, 50 loops each)
